# Train document-denoising models on Kaggle

Before running any cell, set these in the notebook's right sidebar:

1. **Settings → Accelerator → GPU T4 x2** (or P100).
2. **Settings → Internet → On** (needed for `git clone`, `pip install`, and the Swin2SR backbone, which pulls pretrained weights from Hugging Face).
3. **+ Add Input → Datasets → search `document-crops`** and attach it. Kaggle's exact mount path can vary (e.g. `/kaggle/input/document-crops/` or `/kaggle/input/datasets/<owner>/document-crops/`) — the auto-detect cell below finds it either way, so don't hardcode a path yourself.
4. **If you attach the dataset while the session is already running, restart the session afterward** — a newly added input doesn't hot-mount, and you'll otherwise see an empty `/kaggle/input`.
5. Optional, for W&B logging: **Add-ons → Secrets → add `WANDB_API_KEY`**, then toggle it "attached to this notebook" (having the secret in your account isn't enough). Without it, training falls back to CSV logging — still works, just no dashboard.

This notebook clones the repo fresh each run, so it always trains whatever is currently pushed to `master` — it never depends on local edits.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — check Settings → Accelerator before continuing.")

In [ ]:
REPO_URL = "https://github.com/jere1882/DocDenoising.git"

!rm -rf DocDenoising
!git clone --depth 1 {REPO_URL}
%cd DocDenoising
!pip install -q -e .
print("install done")

In [ ]:
# Auto-detect where the clean crops actually live under /kaggle/input.
# Deliberately not hardcoding the nesting (e.g. /kaggle/input/<slug>/ vs
# /kaggle/input/datasets/<owner>/<slug>/) since Kaggle's mount layout can vary —
# search everything under /kaggle/input instead.
import glob
import os

pngs = glob.glob("/kaggle/input/**/*.png", recursive=True)
assert pngs, "No .png files found under /kaggle/input — did you attach the dataset in '+ Add Input', and restart the session afterward?"

CLEAN_DIR = os.path.dirname(pngs[0])
print(f"found {len(pngs)} crops, using CLEAN_DIR = {CLEAN_DIR}")

In [ ]:
# Optional W&B login via a Kaggle Secret. Falls back to CSV logging if not set.
LOGGER = "csv"
try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    import wandb
    wandb.login(key=wandb_key)
    LOGGER = "wandb"
    print("W&B login ok — using logger=wandb")
except Exception as e:
    print(f"No WANDB_API_KEY secret found ({e!r}) — using logger=csv")

In [ ]:
MODEL = "residual_unet"   # unet | plain_encoder_decoder | residual_unet | swin2sr
MAX_EPOCHS = 20
RUN_DIR = "/kaggle/working/outputs/run1"

# All three +trainer.* keys are NEW keys not in conf/trainer/default.yaml
# (hence '+' — Hydra's structured configs reject unknown keys without it).
# - precision=16-mixed: uses the T4's Tensor Cores (default runs full FP32).
# - devices=2 + strategy=ddp_notebook: use both GPUs. ddp_notebook (not plain
#   ddp) is required because plain DDP assumes it can relaunch the script as
#   separate processes, which doesn't work inside a live notebook kernel.
# - sync_batchnorm=true: REQUIRED alongside devices=2. Every DoubleConv block
#   uses BatchNorm2d, whose running stats are buffers, not gradients — DDP's
#   gradient all-reduce does not sync them. Without this flag each GPU's
#   BatchNorm drifts independently, which manifests as val_psnr/val_loss
#   oscillating without improving even while train_loss drops normally.
!denoising-train \
    model={MODEL} \
    data.clean_dir={CLEAN_DIR} \
    hydra.run.dir={RUN_DIR} \
    trainer.max_epochs={MAX_EPOCHS} \
    trainer.devices=2 \
    +trainer.strategy=ddp_notebook \
    +trainer.precision=16-mixed \
    +trainer.sync_batchnorm=true \
    logger={LOGGER}

In [ ]:
# Package the checkpoints so they show up under this notebook's Output tab for download.
import shutil

ckpt_dir = f"{RUN_DIR}/checkpoints"
out_zip = "/kaggle/working/checkpoints"
shutil.make_archive(out_zip, "zip", ckpt_dir)
print(f"saved {out_zip}.zip")

## Notes

- **To keep training after closing the browser tab**: use *Save Version → Save & Run All (Commit)* instead of running cells interactively. Everything under `/kaggle/working/` persists in the committed version's Output tab.
- **To resume a run**: re-attach the run's checkpoint (as a Kaggle Dataset, or re-upload) and add `ckpt_path=<path>/last.ckpt` to the `denoising-train` overrides above.
- **GPU quota**: 30 GPU-hours/week, visible under Settings. A 20-epoch run at 512px with `batch_size=8` took roughly 4–4.5 hours on a T4 for the equivalent local runs — budget accordingly.
- **Swin2SR**: needs Internet on (see top of notebook) to pull pretrained weights from `caidas/swin2SR-lightweight-x2-64` on first use.